In [1]:
# ── 0. Imports ────────────────────────────────────────────────────────────
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import defaultdict
 
import torch
from transformers import BertTokenizerFast, BertModel
 
from scipy.spatial.distance import cosine
from scipy.stats import bootstrap
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
 
warnings.filterwarnings("ignore")

# ── re-import everything from your main file ──────────────────────────────
# ── Configuration (mirrors bert_domain_drift.py) ──────────────────────────
TARGET_PUBMED   = ["neural", "memory", "cell", "agent", "network", "node"]
TARGET_GUARDIAN = ["cloud", "stream","port"]

# Period labels exactly as they will appear in the output table.
# Key   = the label string printed in the table
# Value = path to the corresponding CSV file
ARXIV_PERIODS_PUBMED = {
    "arxiv_1990_2011_p": "./filtered/arxiv_1990_2011_pubmed.csv",
    "arxiv_2012_2019_p": "./filtered/arxiv_2012_2019_pubmed.csv",
    "arxiv_2020_2023_p": "./filtered/arxiv_2020_2023_pubmed.csv",
}

ARXIV_PERIODS_GUARDIAN = {
    "arxiv_1990_2011_g": "./filtered/arxiv_1990_2011_guardian.csv",
    "arxiv_2012_2019_g": "./filtered/arxiv_2012_2019_guardian.csv",
    "arxiv_2020_2023_g": "./filtered/arxiv_2020_2023_guardian.csv",
}

# How many sentences to sample per word per corpus.
N_SAMPLES = 500
 
# BERT variant — bert-base-uncased is fine for this task.
BERT_MODEL = "bert-base-uncased"
 
# Which BERT layer(s) to use for the representation.
LAYER_STRATEGY = "last"
 
# Batch size for BERT inference — reduce if you hit OOM on CPU
BATCH_SIZE = 16
 
# Random seed for reproducible sampling
SEED = 42
rng = np.random.default_rng(SEED)
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [9]:
# ── 3. Sentence sampling ──────────────────────────────────────────────────
 
def sentences_containing(word, corpus, n=N_SAMPLES, seed=SEED):
    """
    Find sentences in the corpus that contain `word` as a whole token
    (case-insensitive, word-boundary matched) and return a random sample.
 
    We split each document into sentences on full-stops/newlines first
    so BERT doesn't have to handle very long inputs.
 
    Why sample rather than use all?  With PubMed having 2.2 M rows and
    BERT taking ~0.1 s per batch, using everything would be hours.
    500 sentences gives a centroid that's stable to < 0.005 cosine units.
    """
    pattern = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
 
    matched = []
    for doc in corpus:
        # Split into crude sentences
        for sent in re.split(r"[.\n]", str(doc)):
            sent = sent.strip()
            if len(sent) > 15 and pattern.search(sent):
                matched.append(sent)
 
    if len(matched) == 0:
        print(f"  WARNING: '{word}' not found in corpus")
        return []
 
    rng_local = np.random.default_rng(seed)
    idx = rng_local.choice(len(matched), size=min(1000, len(matched)), replace=False)
    sampled = [matched[i] for i in idx]
    print(f"  '{word}': {len(matched):,} hits → sampling {len(sampled)}")
    return sampled

# ── 5. Centroid computation and drift measurement ─────────────────────────
 
def compute_centroid(embeddings):
    """L2-normalise each vector then average. This is the standard approach
    for comparing directional similarity in high-dim spaces."""
    normed = normalize(embeddings, norm="l2")
    centroid = normed.mean(axis=0)
    # Normalise the centroid itself so cosine distance is interpretable
    return centroid / np.linalg.norm(centroid)

 
def bootstrap_intra_variance(embeddings, n_bootstrap=500, subsample=250):
    """
    Noise floor: how much does the centroid wobble when we resample
    within the *same* corpus?  Any cross-domain distance above this is
    meaningful drift, not just sampling noise.
 
    We split embeddings in half randomly n_bootstrap times and compute
    the cosine distance between the two half-centroids.  The mean of
    those distances is epsilon, the intra-corpus noise floor.
 
    This is the BERT analogue of your Procrustes SNR epsilon.
    """
    n = len(embeddings)
    if n < 50:
        return np.nan, np.nan
 
    sub = min(subsample, n // 2)
    dists = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=sub * 2, replace=False)
        half_a = embeddings[idx[:sub]]
        half_b = embeddings[idx[sub:]]
        dists.append(cosine_dist(compute_centroid(half_a), compute_centroid(half_b)))
 
    return float(np.mean(dists)), float(np.std(dists))

In [10]:
# ── 4. BERT embedding extraction ─────────────────────────────────────────
 
tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL)
model = BertModel.from_pretrained(BERT_MODEL, output_hidden_states=True)
model.eval()
model.to(DEVICE)
print(f"BERT model '{BERT_MODEL}' loaded")

def get_word_embedding(sentences, target_word, batch_size=BATCH_SIZE):
    """
    For each sentence, extract the contextual embedding of `target_word`.
 
    Because BERT uses WordPiece tokenization, 'neural' stays as one token
    but 'hallucination' might split into ['hall', '##uci', '##nation'].
    We handle this by averaging the sub-token vectors that make up the word.
 
    Returns a (N, 768) numpy array where N = number of sentences where
    the target word was actually found after tokenisation.
    (Very rarely the pattern matches in the raw text but the tokenizer
    doesn't produce a clean match — those sentences are skipped.)
 
    Layer strategy:
      "last"  → single final layer embedding
      "mean4" → mean of last 4 hidden layers (richer representation)
    """
    target_lower = target_word.lower()
    embeddings = []
 
    for batch_start in range(0, len(sentences), batch_size):
        batch = sentences[batch_start : batch_start + batch_size]
 
        # Tokenize — truncate to 512 tokens (BERT's max)
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
            return_offsets_mapping=False,
        )
        
        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)
 
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
 
        
        hidden_states = outputs.hidden_states  # len = 13 for bert-base
 
        if LAYER_STRATEGY == "last":
            token_vecs = hidden_states[-1]          # (B, T, 768)
        else:  # mean4
            token_vecs = torch.stack(hidden_states[-4:], dim=0).mean(dim=0)
 
        token_vecs = token_vecs.cpu().numpy()
 
        for sent_idx, sent in enumerate(batch):
            token_ids_row = input_ids[sent_idx].cpu().tolist()
            tokens = tokenizer.convert_ids_to_tokens(token_ids_row)
            target_chars = target_lower
            found_indices = []
 
            i = 0
            while i < len(tokens):
                # Try to match starting at position i
                reconstructed = ""
                span = []
                j = i
                while j < len(tokens) and len(reconstructed) <= len(target_chars):
                    tok = tokens[j]
                    if tok in ("[PAD]", "[CLS]", "[SEP]"):
                        break
                    # Strip the ## continuation marker
                    clean_tok = tok.lstrip("#")
                    reconstructed += clean_tok
                    span.append(j)
                    if reconstructed == target_chars:
                        found_indices = span
                        break
                    if not target_chars.startswith(reconstructed):
                        break
                    j += 1
                if found_indices:
                    break
                i += 1
 
            if not found_indices:
                continue  # target word not found in tokenized form
 
            word_vec = token_vecs[sent_idx, found_indices, :].mean(axis=0)
            embeddings.append(word_vec)
 
    if len(embeddings) == 0:
        return None
    return np.vstack(embeddings)  # (N, 768)


BERT model 'bert-base-uncased' loaded


In [11]:
# ── Helpers ───────────────────────────────────────────────────────────────

def get_corpus_centroid(word, corpus, corpus_label, n=500):
    """
    Extract BERT embeddings for `word` in `corpus` and return the centroid.
    Returns (None, 0) if the word is not found or has too few examples.
    Caller is responsible for caching if the same corpus is reused.
    """
    sents = sentences_containing(word, corpus, n=n)
    if not sents:
        print(f"    '{word}' not found in {corpus_label}")
        return None, 0

    t0 = time.time()
    emb = get_word_embedding(sents, word)
    elapsed = time.time() - t0

    if emb is None or len(emb) < 10:
        print(f"    '{word}' in {corpus_label}: too few tokenised hits")
        return None, 0

    print(f"    '{word}' in {corpus_label}: {len(emb)} embeddings ({elapsed:.1f}s)")
    return compute_centroid(emb), len(emb)

def load_period_csv(path, col="text"):
    df = pd.read_csv(path, usecols=[col]).dropna(subset=[col])
    print(f"  Loaded {path}: {len(df):,} rows")
    return df[col].tolist()

In [12]:
def cosine_dist(a, b):
    """1 - cosine_similarity.  0 = identical direction, 2 = opposite."""
    return float(cosine(a, b))


In [13]:
# ── Main period-drift function ────────────────────────────────────────────

def run_period_drift(
    pubmed_corpus,
    guardian_corpus,
    arxiv_periods_pubmed   = ARXIV_PERIODS_PUBMED,
    arxiv_periods_guardian = ARXIV_PERIODS_GUARDIAN,
    target_pubmed          = TARGET_PUBMED,
    target_guardian        = TARGET_GUARDIAN,
    n_samples              = 500,
):
    """
    For each word x arXiv period, compute cosine distance between the
    arXiv-period centroid and the fixed comparison-corpus centroid.

    The PubMed / Guardian centroid is computed ONCE per word and reused
    across all three arXiv periods -- no redundant BERT inference.

    Returns
    -------
    pd.DataFrame with columns:
        word, arxiv_period, vs_corpus, bert_dist, n_arxiv, n_other
    """
    rows = []

    # ── BLOCK 1: arXiv periods vs PubMed ─────────────────────────────────
    print("\n" + "="*60)
    print("BLOCK 1: arXiv periods vs PubMed")
    print("="*60)

    for word in target_pubmed:
        print(f"\n── {word} ──────────────────────────────")

        # Compute PubMed centroid once for this word
        print(f"  [PubMed centroid]")
        pm_centroid, n_pm = get_corpus_centroid(
            word, pubmed_corpus, "pubmed", n=n_samples
        )
        if pm_centroid is None:
            continue

        for period_label, csv_path in arxiv_periods_pubmed.items():
            print(f"  [arXiv period: {period_label}]")
            try:
                period_corpus = load_period_csv(csv_path)
            except FileNotFoundError:
                print(f"    File not found: {csv_path} -- skipping")
                continue

            ax_centroid, n_ax = get_corpus_centroid(
                word, period_corpus, period_label, n=n_samples
            )
            if ax_centroid is None:
                continue

            dist = cosine_dist(ax_centroid, pm_centroid)
            print(f"    {period_label} vs pubmed: {dist:.5f}")

            rows.append({
                "word":         word,
                "arxiv_period": period_label,
                "vs_corpus":    "pubmed",
                "bert_dist":    round(dist, 5),
                "n_arxiv":      n_ax,
                "n_other":      n_pm,
            })

    # ── BLOCK 2: arXiv periods vs Guardian ───────────────────────────────
    print("\n" + "="*60)
    print("BLOCK 2: arXiv periods vs Guardian")
    print("="*60)

    for word in target_guardian:
        print(f"\n── {word} ──────────────────────────────")

        # Compute Guardian centroid once for this word
        print(f"  [Guardian centroid]")
        gd_centroid, n_gd = get_corpus_centroid(
            word, guardian_corpus, "guardian", n=n_samples
        )
        if gd_centroid is None:
            continue

        for period_label, csv_path in arxiv_periods_guardian.items():
            print(f"  [arXiv period: {period_label}]")
            try:
                period_corpus = load_period_csv(csv_path)
            except FileNotFoundError:
                print(f"    File not found: {csv_path} -- skipping")
                continue

            ax_centroid, n_ax = get_corpus_centroid(
                word, period_corpus, period_label, n=n_samples
            )
            if ax_centroid is None:
                continue

            dist = cosine_dist(ax_centroid, gd_centroid)
            print(f"    {period_label} vs guardian: {dist:.5f}")

            rows.append({
                "word":         word,
                "arxiv_period": period_label,
                "vs_corpus":    "guardian",
                "bert_dist":    round(dist, 5),
                "n_arxiv":      n_ax,
                "n_other":      n_gd,
            })

    return pd.DataFrame(rows)

In [14]:
def print_drift_table(df):
    if df.empty:
        print("No results.")
        return

    header = f"{'word':>8}  {'arxiv_period':<22}  {'vs_corpus':<10}  {'bert_dist':>10}"
    print("\n" + header)
    print("-" * len(header))
    for _, row in df.iterrows():
        print(
            f"{row['word']:>8}  "
            f"{row['arxiv_period']:<22}  "
            f"{row['vs_corpus']:<10}  "
            f"{row['bert_dist']:>10.5f}"
        )

In [15]:
# ── Entry point ───────────────────────────────────────────────────────────

# Load fixed comparison corpora
print("Loading PubMed corpus...")
pubmed_corpus = load_period_csv(
    "./filtered/pubmed.csv", col="abstract_text"
)

print("Loading Guardian corpus...")
guardian_corpus = load_period_csv(
    "./filtered/guardian.csv", col="bodyContent"
)

# Run the period-wise drift
results = run_period_drift(
    pubmed_corpus   = pubmed_corpus,
    guardian_corpus = guardian_corpus,
)

# Display and save
print_drift_table(results)
# results.to_csv("bert_period_drift_results.csv", index=False)
# print("\nSaved: bert_period_drift_results.csv")

Loading PubMed corpus...
  Loaded ./filtered/pubmed.csv: 30,540 rows
Loading Guardian corpus...
  Loaded ./filtered/guardian.csv: 6,596 rows

BLOCK 1: arXiv periods vs PubMed

── neural ──────────────────────────────
  [PubMed centroid]
  'neural': 913 hits → sampling 913
    'neural' in pubmed: 913 embeddings (3.7s)
  [arXiv period: arxiv_1990_2011_p]
  Loaded ./filtered/arxiv_1990_2011_pubmed.csv: 3,673 rows
  'neural': 514 hits → sampling 514
    'neural' in arxiv_1990_2011_p: 514 embeddings (1.8s)
    arxiv_1990_2011_p vs pubmed: 0.14900
  [arXiv period: arxiv_2012_2019_p]
  Loaded ./filtered/arxiv_2012_2019_pubmed.csv: 20,317 rows
  'neural': 10,085 hits → sampling 1000
    'neural' in arxiv_2012_2019_p: 1000 embeddings (3.7s)
    arxiv_2012_2019_p vs pubmed: 0.15306
  [arXiv period: arxiv_2020_2023_p]
  Loaded ./filtered/arxiv_2020_2023_pubmed.csv: 38,448 rows
  'neural': 26,069 hits → sampling 1000
    'neural' in arxiv_2020_2023_p: 1000 embeddings (4.1s)
    arxiv_2020_2023_p v